# Phase 2 LNER: LoRA on my phase 1 gqa_swa model

Attach the phase 1 output (`gqa_swa_best.pt` and `bpe_legal_16k_v2.pkl`) and add the `HF` secret.

In [ ]:
import os, sys, json, math, random, time, zipfile, glob, re, bisect, pickle, gc
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint as grad_ckpt
from tqdm.auto import tqdm
from huggingface_hub import login, hf_hub_download
from types import SimpleNamespace

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    assert HF_TOKEN, "HF token not found"
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Logged in.")

## Config

In [ ]:
# extra penalty on the O tag, tried on dev (more = more recall)
DELTAS = [0.0, -0.5, 0.5, 1.0, 1.5, 2.0, 3.0]

USE_CRF = True
# the causal lm is run with bidirectional attention for tagging
BIDIRECTIONAL = True
BIDIR_WINDOW = 256
PARTIAL_LAYERS = 6
RANK, ALPHA = 8, 16.0
# docs are cut into overlapping windows of 512 tokens
WIDTH, STRIDE = 512, 256
# docs of fold2 kept aside as dev
DEV_COUNT = 8
INFER_BATCH = 16
BATCH = 8
EPOCHS = 8
PATIENCE = 8
LR = 3e-4
HEAD_LR = 5e-4
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
WEIGHT_CAP = 15.0
USE_GC = False
SEED = 42
SMOKE = False

LORMA_RANK, LORMA_ALPHA = 16, 32.0
FULL_LR = 2e-5
STUDENT_RANK, STUDENT_ALPHA = 4, 8.0
KD_TEMP = 3.0
KD_WEIGHT = 0.5
EXPERIMENTS = [
    {"num": 1, "name": "lora", "method": "lora", "rank": RANK, "alpha": ALPHA, "lr": LR},
    {"num": 2, "name": "lorma", "method": "lorma", "rank": LORMA_RANK, "alpha": LORMA_ALPHA, "lr": LR},
    {"num": 3, "name": "full", "method": "full", "rank": None, "alpha": None, "lr": FULL_LR},
    {"num": 4, "name": "distillation", "method": "lora", "rank": STUDENT_RANK, "alpha": STUDENT_ALPHA, "lr": LR, "kd": True},
]

# None = look for them in /kaggle/input
CKPT_PATH = None
TOK_PATH = None

# not stored in the checkpoint, same as phase 1
MAX_SEQ_LEN = 1023
ATTN_WINDOW, GLOBAL_EVERY = 256, 4

OUT_DIR = "/kaggle/working/lner_lora"
os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev, "|", torch.cuda.get_device_name(0) if dev.type == "cuda" else "")

## Phase 1 checkpoint and tokenizer

In [ ]:
# take the checkpoint with the highest step if there are several
if CKPT_PATH is None:
    hits = glob.glob("/kaggle/input/**/gqa_swa_best.pt", recursive=True)
    print("checkpoints found:", hits)
    if hits:
        CKPT_PATH = hits[0] if len(hits) == 1 else max(hits, key=lambda p: torch.load(p, map_location="cpu", weights_only=False).get("step", 0))
assert CKPT_PATH and os.path.isfile(CKPT_PATH), "gqa_swa_best.pt not found in /kaggle/input"

# tokenizer that sits closest to the checkpoint in the folder tree
if TOK_PATH is None:
    toks = glob.glob("/kaggle/input/**/bpe_legal_16k_v2.pkl", recursive=True)
    print("tokenizers found:", toks)
    if toks:
        TOK_PATH = max(toks, key=lambda t: len(os.path.commonpath([CKPT_PATH, t])))
assert TOK_PATH and os.path.isfile(TOK_PATH), "bpe_legal_16k_v2.pkl not found in /kaggle/input"
print("checkpoint:", CKPT_PATH)
print("tokenizer :", TOK_PATH)


class CustomBPETokenizer:
    def __init__(self, vocab_size=16384):
        self.target_vocab_size = vocab_size
        self.special_tokens = ["<pad>", "<unk>", "<s>", "</s>"]
        self.pad_token_id = 0
        self.unk_token_id = 1
        self.bos_token_id = 2
        self.eos_token_id = 3
        self.pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\w+| ?\S|\s+(?!\S)|\s+""")
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = {}
        self._word_cache = {}

    def _tokenize_word(self, word):
        word_tuple = tuple(word)
        if len(word_tuple) <= 1:
            return list(word_tuple)
        while True:
            pairs = [(word_tuple[i], word_tuple[i + 1]) for i in range(len(word_tuple) - 1)]
            valid_merges = [p for p in pairs if p in self.merges]
            if not valid_merges:
                break
            best_pair = min(valid_merges, key=lambda p: self.vocab[self.merges[p]])
            first, second = best_pair
            merged_tok = self.merges[best_pair]
            new_tuple = []
            i = 0
            while i < len(word_tuple):
                if i < len(word_tuple) - 1 and word_tuple[i] == first and word_tuple[i + 1] == second:
                    new_tuple.append(merged_tok)
                    i += 2
                else:
                    new_tuple.append(word_tuple[i])
                    i += 1
            word_tuple = tuple(new_tuple)
            if len(word_tuple) == 1:
                break
        return list(word_tuple)

    def encode(self, text, add_special_tokens=False):
        if not text:
            return []
        cache = self._word_cache
        ids = []
        for word in self.pat.findall(text):
            word_ids = cache.get(word)
            if word_ids is None:
                word_ids = [self.vocab.get(tok, self.unk_token_id) for tok in self._tokenize_word(word)]
                if len(cache) < 2_000_000:
                    cache[word] = word_ids
            ids.extend(word_ids)
        if add_special_tokens:
            ids = [self.bos_token_id] + ids + [self.eos_token_id]
        return ids

    @property
    def vocab_size(self):
        return len(self.vocab)

    @classmethod
    def load(cls, filepath):
        with open(filepath, "rb") as f:
            data = pickle.load(f)
        tok = cls(vocab_size=data["target_vocab_size"])
        tok.vocab = data["vocab"]
        tok.inverse_vocab = data["inverse_vocab"]
        tok.merges = data["merges"]
        tok.special_tokens = data["special_tokens"]
        print(f"Loaded tokenizer from {filepath} (Vocab size: {tok.vocab_size})")
        return tok


tokenizer = CustomBPETokenizer.load(TOK_PATH)
VOCAB_SIZE = tokenizer.vocab_size

_WORD_CACHE = {}


# same ids as tokenizer.encode, plus the char (start, end) of every token
def tokenize_with_offsets(text):
    ids, offs = [], []
    for mt in tokenizer.pat.finditer(text):
        w = mt.group(0)
        got = _WORD_CACHE.get(w)
        if got is None:
            toks = tokenizer._tokenize_word(w)
            got = ([tokenizer.vocab.get(t, tokenizer.unk_token_id) for t in toks], [len(t) for t in toks])
            if len(_WORD_CACHE) < 2_000_000:
                _WORD_CACHE[w] = got
        pos = mt.start()
        for i, ln in zip(*got):
            ids.append(i)
            offs.append((pos, pos + ln))
            pos += ln
    return ids, offs

## Data

In [ ]:
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
P2 = "Phase_2_LNER"
LABELS = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
LABEL_SET = set(LABELS)
# O, then B-x and I-x for every label
TAG_NAMES = ["O"] + [f"{p}-{l}" for l in LABELS for p in ("B", "I")]
K = len(TAG_NAMES)


def fetch(rel):
    return hf_hub_download(repo_id=HF_DATASET_REPO, filename=rel, repo_type="dataset", token=HF_TOKEN)


def read_json(path):
    with open(path, "r", encoding="utf-8") as fh:
        return json.load(fh)


# reads one fold and drops entities with a bad label or span
def load_fold(text_rel, gold_rel=None):
    texts = read_json(fetch(text_rel))
    golds = read_json(fetch(gold_rel)) if gold_rel else {}
    rows, total, dropped, mismatch = [], 0, 0, 0
    for doc_id, text in texts.items():
        ents = []
        for it in golds.get(doc_id, []):
            # a gold entry is [start, end, label, text]
            s, e, lab = int(it[0]), int(it[1]), str(it[2])
            total += 1
            # keep only known labels and spans inside the text
            if lab in LABEL_SET and 0 <= s < e <= len(text):
                ents.append((s, e, lab))
                if len(it) > 3 and text[s:e] != it[3]:
                    mismatch += 1
            else:
                dropped += 1
        rows.append((str(doc_id), text, sorted(ents)))
    if gold_rel:
        print(f"{gold_rel}: {len(rows)} docs, {total} entities, dropped {dropped}, text[s:e] != entity_text for {mismatch}")
    return rows


fold1 = load_fold(f"{P2}/Train/fold1-text.json", f"{P2}/Train/fold1-gold.json")
fold2 = load_fold(f"{P2}/Train/fold2-text.json", f"{P2}/Train/fold2-gold.json")
# test has no gold file
test_rows = load_fold(f"{P2}/Test/fold3-text.json")
print("fold1", len(fold1), "fold2", len(fold2), "test", len(test_rows))

cnt = Counter(l for _, _, ents in fold1 + fold2 for _, _, l in ents)
print("entities per label:", {l: cnt[l] for l in LABELS})

## Utilities

In [ ]:
# token ids and offsets, spaces trimmed off each token
def encode_doc(text):
    ids, raw_offs = tokenize_with_offsets(text)
    offs = []
    # trim whitespace from both ends of every token
    for s, e in raw_offs:
        while s < e and text[s].isspace():
            s += 1
        while e > s and text[e - 1].isspace():
            e -= 1
        offs.append((s, e))
    return ids, offs


# a token takes the entity it overlaps most, B for the first token and I after that
def bio_tags(offs, ents):
    n = len(offs)
    starts = [o[0] for o in offs]
    ends = [o[1] for o in offs]
    # owner[i] = entity that token i belongs to, -1 = none
    owner = [-1] * n
    best = [0.0] * n
    for ei, (s, e, _lab) in enumerate(ents):
        # first token that ends after the entity starts
        i = bisect.bisect_right(ends, s)
        while i < n and starts[i] < e:
            # overlap in characters, the biggest overlap wins
            if ends[i] > starts[i]:
                ov = min(ends[i], e) - max(starts[i], s)
            else:
                ov = 0.5 if (s < starts[i] < e) else 0.0
            if ov > best[i]:
                best[i], owner[i] = ov, ei
            i += 1
    # owners -> tag ids: 0 = O, 1+2i = B, 2+2i = I
    tags = [0] * n
    for i, ei in enumerate(owner):
        if ei < 0:
            continue
        li = LABELS.index(ents[ei][2])
        tags[i] = 2 + 2 * li if (i > 0 and owner[i - 1] == ei) else 1 + 2 * li
    return tags


# overlapping (start, end) windows over n tokens
def window_ranges(n, width, stride):
    if n == 0:
        return []
    out, a = [], 0
    while True:
        b = min(a + width, n)
        out.append((a, b))
        if b == n:
            return out
        a += stride


# mode 0 = as is, 1 = strip trailing , ; : , 2 = also a trailing dot
def trim_span(text, s, e, mode):
    if mode == 0:
        return s, e
    while e > s and (text[e - 1] in ",;:" or text[e - 1].isspace()):
        e -= 1
    while s < e and text[s].isspace():
        s += 1
    if mode == 2 and e - s >= 2 and text[e - 1] == "." and text[e - 2].isalnum():
        if not text[s:e].endswith((" v.", " vs.", " No.", " Ltd.", " Pvt.", " Dr.", " Mr.", " Ms.", " Mrs.", " Co.", " Inc.")):
            e -= 1
    return s, e


def decode_spans(text, offs, path, trim_mode=0):
    spans, seen, i, n = [], set(), 0, len(path)
    while i < n:
        t = int(path[i])
        if t == 0:
            i += 1
            continue
        li = (t - 1) // 2
        j = i + 1
        # keep going while the next tags are I of the same label
        while j < n and int(path[j]) == 2 + 2 * li:
            j += 1
        # empty tokens are skipped when taking the char span
        nz = [k for k in range(i, j) if offs[k][1] > offs[k][0]]
        if nz:
            s, e = trim_span(text, offs[nz[0]][0], offs[nz[-1]][1], trim_mode)
            sp = (s, e, LABELS[li])
            if e > s and sp not in seen:
                seen.add(sp)
                spans.append(sp)
        i = j
    return spans


# blocks illegal moves like O -> I-X
def allowed_matrix(tag_names):
    k = len(tag_names)
    allowed = np.zeros((k, k), dtype=bool)
    for a, sa in enumerate(tag_names):
        for b, sb in enumerate(tag_names):
            if sb == "O" or sb.startswith("B-"):
                allowed[a, b] = True
            else:
                allowed[a, b] = sa in ("B-" + sb[2:], "I-" + sb[2:])
    return allowed


ALLOWED = allowed_matrix(TAG_NAMES)
HARD_TRANS = np.where(ALLOWED, 0.0, -1e4)  # used when there is no crf


def log_softmax_np(x):
    x = x - x.max(axis=-1, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=-1, keepdims=True))


# best tag path from emission and transition scores
def viterbi(em, trans):
    T, k = em.shape
    if T == 0:
        return []
    score = em[0].copy()
    # back[t, j] = best previous tag for tag j at step t
    back = np.zeros((T, k), dtype=np.int32)
    cols = np.arange(k)
    for t in range(1, T):
        cand = score[:, None] + trans
        idx = cand.argmax(axis=0)
        back[t] = idx
        score = cand[idx, cols] + em[t]
    # walk back from the best last tag
    cur = int(score.argmax())
    path = [cur]
    for t in range(T - 1, 0, -1):
        cur = int(back[t, cur])
        path.append(cur)
    return path[::-1]


def best_path(logits, trans, use_crf, o_delta=0.0):
    em = logits.astype(np.float64) if use_crf else log_softmax_np(logits.astype(np.float64))
    # lower the score of O, so more entities get predicted
    em[:, 0] -= o_delta
    return viterbi(em, trans)


# exact span + label match, macro average over the 12 labels
def strict_scores(gold_rows, preds):
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc_id, _text, ents in gold_rows:
        # count tp / fp / fn per label on exact (start, end, label) matches
        g = set(ents)
        p = set(tuple(x[:3]) for x in preds.get(doc_id, []))
        for l in LABELS:
            gl = {x for x in g if x[2] == l}
            pl = {x for x in p if x[2] == l}
            tp[l] += len(gl & pl)
            fp[l] += len(pl - gl)
            fn[l] += len(gl - pl)
    per, ps, rs, fs = {}, [], [], []
    for l in LABELS:
        pr = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        rc = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        f1 = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
        per[l] = (pr, rc, f1, tp[l], fp[l], fn[l])
        ps.append(pr); rs.append(rc); fs.append(f1)
    return {"f1": float(np.mean(fs)), "p": float(np.mean(ps)), "r": float(np.mean(rs)), "per": per}


def print_scores(sc):
    for l in LABELS:
        pr, rc, f1, tp_, fp_, fn_ = sc["per"][l]
        print(f"  {l:10s} P={pr:.3f} R={rc:.3f} F1={f1:.3f}  (tp {tp_}, fp {fp_}, fn {fn_})")
    print(f"  MACRO      P={sc['p']:.3f} R={sc['r']:.3f} F1={sc['f1']:.3f}")

## Offset check

In [ ]:
PREP = {}


def prepare(rows):
    for doc_id, text, _ in tqdm(rows, desc="tokenizing", leave=False):
        if doc_id not in PREP:
            PREP[doc_id] = encode_doc(text)


all_train = fold1 + fold2
for _d, _t, _ in all_train[:5]:
    assert tokenize_with_offsets(_t)[0] == tokenizer.encode(_t)
print("offset tokenizer matches encode")
prepare(all_train)
# tokens per doc
lens = np.array([len(PREP[d][0]) for d, _, _ in all_train])
print(f"tokens/doc: mean {lens.mean():.0f}, median {np.median(lens):.0f}, max {lens.max()}; docs longer than WIDTH: {(lens > WIDTH).mean():.0%}")

for doc_id, text, _ in all_train[:20]:
    ids, offs = PREP[doc_id]
    assert all(0 <= s <= e <= len(text) for s, e in offs)
    assert all(a[0] <= b[0] and a[1] <= b[1] for a, b in zip(offs, offs[1:]))

covered = total = shown = 0
for doc_id, text, ents in all_train:
    ids, offs = PREP[doc_id]
    # can every gold span start and end on a token boundary?
    st = {s for s, e in offs if e > s}
    en = {e for s, e in offs if e > s}
    for s, e, l in ents:
        total += 1
        if s in st and e in en:
            covered += 1
        elif shown < 8:
            shown += 1
            print(f"  off-boundary {l}: {text[s:e][:50]!r} in ...{text[max(0, s - 10):e + 10][:80]!r}")
print(f"gold spans on token boundaries: {covered}/{total} = {covered / max(total, 1):.3f}")

# upper bound: gold tags -> spans again, used to pick the trim mode
oracle = {}
for mode in (0, 1, 2):
    preds = {d: decode_spans(t, PREP[d][1], bio_tags(PREP[d][1], ents), mode) for d, t, ents in all_train}
    oracle[mode] = strict_scores(all_train, preds)
    print(f"oracle round-trip, trim mode {mode}: macro-F1 {oracle[mode]['f1']:.4f}")
# best trim mode, a tie goes to the smaller one
TRIM_MODE = max((0, 1, 2), key=lambda m: (oracle[m]["f1"], -m))
print("TRIM_MODE =", TRIM_MODE)
print_scores(oracle[TRIM_MODE])

## Phase 1 model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * (x * torch.rsqrt(variance + self.eps))


class SwiGLU(nn.Module):
    def __init__(self, in_features, hidden_features):
        super().__init__()
        self.w1 = nn.Linear(in_features, hidden_features, bias=False)
        self.w2 = nn.Linear(in_features, hidden_features, bias=False)
        self.w3 = nn.Linear(hidden_features, in_features, bias=False)

    def forward(self, x):
        return self.w3(F.silu(self.w1(x)) * self.w2(x))


_ROPE = {}


def _rope_tables(head_dim, device, max_len=2048):
    key = (head_dim, str(device))
    if key not in _ROPE:
        # rotary position tables
        inv_freq = 1.0 / (10000 ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
        t = torch.arange(max_len, device=device).float()
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        _ROPE[key] = (emb.cos(), emb.sin())
    return _ROPE[key]


def rotate_half(x):
    x1, x2 = x[..., : x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k):
    T, hd = q.size(-2), q.size(-1)
    cos, sin = _rope_tables(hd, q.device)
    cos = cos[:T].to(q.dtype)[None, None]
    sin = sin[:T].to(q.dtype)[None, None]
    return q * cos + rotate_half(q) * sin, k * cos + rotate_half(k) * sin


_MASK = {}


def causal_window_mask(T, window, device):
    key = (T, window, str(device))
    m = _MASK.get(key)
    if m is None:
        _MASK.clear()
        i = torch.arange(T, device=device)[:, None]
        j = torch.arange(T, device=device)[None, :]
        # look back only, and only `window` tokens
        m = (j <= i) & ((i - j) < window)
        _MASK[key] = m
    return m


def attend(q, k, v, window=None):
    T = q.size(-2)
    # global layer or short input: plain causal attention
    if window is None or window >= T:
        return F.scaled_dot_product_attention(q, k, v, is_causal=True)
    return F.scaled_dot_product_attention(q, k, v, attn_mask=causal_window_mask(T, window, q.device))


def init_weights(model, n_layers):
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    model.apply(_init)
    # smaller init for the output projections
    for name, p in model.named_parameters():
        if name.endswith("o_proj.weight") or name.endswith("w3.weight"):
            nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))


# fewer k/v heads than q heads (grouped query)
class GQACausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, window=None):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads
        self.num_queries_per_kv = n_heads // n_kv_heads
        self.window = window
        self.q_proj = nn.Linear(d_model, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k)
        # repeat k and v so every group of q heads gets a copy
        k = k.repeat_interleave(self.num_queries_per_kv, dim=1)
        v = v.repeat_interleave(self.num_queries_per_kv, dim=1)
        y = attend(q, k, v, self.window)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.o_proj(y)


class _CausalLMBase(nn.Module):
    def __init__(self, vocab_size, d_model, layers, max_seq_len):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.use_grad_ckpt = True
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList(layers)
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        # tied weights
        self.lm_head.weight = self.tok_emb.weight
        init_weights(self, len(layers))

    def forward(self, idx, labels=None):
        x = self.tok_emb(idx)
        for layer in self.layers:
            if self.use_grad_ckpt and self.training:
                x = grad_ckpt(layer, x, use_reentrant=False)
            else:
                x = layer(x)
        x = self.final_norm(x)
        if labels is None:
            x = x[:, -1:, :]
        logits = self.lm_head(x)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), labels.reshape(-1))
        return SimpleNamespace(logits=logits, loss=loss)


class EfficientDecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, window=None):
        super().__init__()
        self.attn_norm = RMSNorm(d_model)
        self.attn = GQACausalAttention(d_model, n_heads, n_kv_heads, window)
        self.ffn_norm = RMSNorm(d_model)
        self.ffn = SwiGLU(d_model, d_ff)

    def forward(self, x):
        x = x + self.attn(self.attn_norm(x))
        return x + self.ffn(self.ffn_norm(x))


class LegalEfficientTransformer(_CausalLMBase):
    def __init__(self, vocab_size, d_model=1024, n_heads=16, n_kv_heads=4, n_layers=16, d_ff=2816,
                 window=ATTN_WINDOW, global_every=4, max_seq_len=MAX_SEQ_LEN):
        # every global_every-th layer is global (window None), the rest use the sliding window
        layers = [EfficientDecoderBlock(d_model, n_heads, n_kv_heads, d_ff,
                                        window=None if (i + 1) % global_every == 0 else window)
                  for i in range(n_layers)]
        super().__init__(vocab_size, d_model, layers, max_seq_len)


# build the phase 1 model from the config saved in the checkpoint
base_ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
cfg_ = dict(base_ck["config"])
base = LegalEfficientTransformer(**cfg_, window=ATTN_WINDOW, global_every=GLOBAL_EVERY)
# remove the torch.compile prefix if it is there
state = {(k[len("_orig_mod."):] if k.startswith("_orig_mod.") else k): v for k, v in base_ck["state_dict"].items()}
missing, unexpected = base.load_state_dict(state, strict=False)
# lm_head is tied to tok_emb so it is not in the file
missing = [k for k in missing if k != "lm_head.weight"]
assert not missing and not unexpected, f"checkpoint mismatch: missing={missing[:5]} unexpected={unexpected[:5]}"
print(f"loaded {CKPT_PATH} (step {base_ck.get('step')}, val loss {base_ck.get('val_loss')})")
print("config:", cfg_)
del base_ck
# the tagger handles checkpointing itself (USE_GC)
base.use_grad_ckpt = False
base = base.to(dev).eval()
# sizes come from the loaded model
D_MODEL, N_LAYERS = base.tok_emb.weight.shape[1], len(base.layers)
assert base.tok_emb.weight.shape[0] == VOCAB_SIZE, "tokenizer vocab size does not match the checkpoint"
assert WIDTH <= base.max_seq_len + 1
print("layer windows:", [l.attn.window for l in base.layers])

def fresh_base():
    m = LegalEfficientTransformer(**cfg_, window=ATTN_WINDOW, global_every=GLOBAL_EVERY)
    m.load_state_dict(state, strict=False)
    m.use_grad_ckpt = False
    return m.to(dev).eval()


## Tagger

In [ ]:
# True = can attend. window mask for the local layers, full mask for the global ones
def build_masks(T, pad_mask, bidirectional, window):
    r = torch.arange(T, device=pad_mask.device).unsqueeze(1)
    c = torch.arange(T, device=pad_mask.device).unsqueeze(0)
    # bidirectional = every token sees both sides
    if bidirectional:
        full = torch.ones(T, T, dtype=torch.bool, device=pad_mask.device)
        win = (c - r).abs() < window
    else:
        full = c <= r
        win = (c <= r) & (c > r - window)
    # never attend to padding
    key = pad_mask[:, None, None, :]
    return full[None, None] & key, win[None, None] & key


# one phase 1 block but with our own mask, so it can run bidirectional
def block_forward(blk, x, mask):
    a = blk.attn
    h = blk.attn_norm(x)
    B, T, C = h.shape
    q = a.q_proj(h).view(B, T, a.n_heads, a.head_dim).transpose(1, 2)
    k = a.k_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    v = a.v_proj(h).view(B, T, a.n_kv_heads, a.head_dim).transpose(1, 2)
    q, k = apply_rope(q, k)
    k = k.repeat_interleave(a.num_queries_per_kv, dim=1)
    v = v.repeat_interleave(a.num_queries_per_kv, dim=1)
    out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask)
    out = out.transpose(1, 2).contiguous().view(B, T, C)
    # residual after attention, then the ffn
    x = x + a.o_proj(out)
    return x + blk.ffn(blk.ffn_norm(x))


# frozen W plus a small trainable B @ A
class LoRALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f, in_f = base_linear.weight.shape
        # B starts at zero so at the start it behaves exactly like phase 1
        self.A = nn.Parameter(torch.randn(rank, in_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        # W x + scale * B A x
        return self.base(x) + self.scale * ((x @ self.A.t()) @ self.B.t())


class LoRMALinear(nn.Module):
    def __init__(self, base_linear, rank, alpha):
        super().__init__()
        self.base = base_linear
        out_f = base_linear.weight.shape[0]
        self.A = nn.Parameter(torch.randn(rank, out_f) * 0.02)
        self.B = nn.Parameter(torch.zeros(out_f, rank))
        self.scale = alpha / rank

    def forward(self, x):
        h = self.base(x)
        return h + self.scale * ((h @ self.A.t()) @ self.B.t())


def adapt_linears(backbone, kind, rank, alpha):
    wrapper = LoRMALinear if kind == "lorma" else LoRALinear
    n = 0
    for blk in backbone.layers:
        # wrap every linear layer of attention and ffn
        for parent in (blk.attn, blk.ffn):
            for name, child in list(parent.named_children()):
                if isinstance(child, nn.Linear):
                    setattr(parent, name, wrapper(child, rank, alpha))
                    n += 1
    return n


# crf over the BIO tags, illegal transitions are blocked
class ChainCRF(nn.Module):
    def __init__(self, tag_names):
        super().__init__()
        self.transitions = nn.Parameter(torch.zeros(len(tag_names), len(tag_names)))
        self.register_buffer("allowed", torch.from_numpy(allowed_matrix(tag_names)))

    def transition(self):
        return self.transitions.masked_fill(~self.allowed, -1e4)

    def nll(self, em, tags, mask):
        trans = self.transition()
        safe = tags.clamp(min=0)
        mf = mask.float()
        emit = em.gather(2, safe.unsqueeze(2)).squeeze(2)
        # score of the gold path
        gold = (emit * mf).sum(1) + (trans[safe[:, :-1], safe[:, 1:]] * mf[:, 1:]).sum(1)
        # forward algorithm for the normaliser
        alpha = em[:, 0]
        for t in range(1, em.size(1)):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + trans.unsqueeze(0) + em[:, t].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, t].unsqueeze(1), nxt, alpha)
        logz = torch.logsumexp(alpha, dim=-1)
        # negative log likelihood per token, averaged
        return ((logz - gold) / mf.sum(1).clamp(min=1)).mean()


class LegalTagger(nn.Module):
    def __init__(self, backbone, tag_names, use_crf, bidirectional, bidir_window, use_gc):
        super().__init__()
        self.backbone = backbone
        self.bidirectional, self.bidir_window, self.use_gc = bidirectional, bidir_window, use_gc
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(D_MODEL, len(tag_names))
        nn.init.normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)
        with torch.no_grad():
            self.head.bias[0] = 2.0  # start off predicting O
        self.crf = ChainCRF(tag_names) if use_crf else None

    def encode(self, ids, pad_mask):
        m = self.backbone
        T = ids.size(1)
        x = m.tok_emb(ids)
        window = self.bidir_window if self.bidirectional else ATTN_WINDOW
        full, win = build_masks(T, pad_mask, self.bidirectional, window)
        for blk in m.layers:
            # local layers get the window mask, global layers the full one
            mask = win if blk.attn.window is not None else full
            if self.use_gc and self.training:
                x = grad_ckpt(block_forward, blk, x, mask, use_reentrant=False)
            else:
                x = block_forward(blk, x, mask)
        return m.final_norm(x)

    def forward(self, ids, pad_mask):
        return self.head(self.drop(self.encode(ids, pad_mask)))


# freeze the phase 1 weights and add the lora adapters
def build_tagger(method, bb, rank, alpha, use_gc):
    for p in bb.parameters():
        # everything frozen unless method is full
        p.requires_grad_(method == "full")
    if method == "partial":
        for blk in bb.layers[-PARTIAL_LAYERS:]:
            for p in blk.parameters():
                p.requires_grad_(True)
        for p in bb.final_norm.parameters():
            p.requires_grad_(True)
    elif method in ("lora", "lorma"):
        print(f"wrapped {adapt_linears(bb, method, rank, alpha)} linear layers with {method}")
    tg = LegalTagger(bb, TAG_NAMES, USE_CRF, BIDIRECTIONAL, BIDIR_WINDOW, use_gc).to(dev)
    tr = sum(p.numel() for p in tg.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in tg.parameters())
    print(f"parameters: trainable {tr / 1e6:.2f}M / total {tot / 1e6:.2f}M ({tr / tot:.1%})")
    return tg

In [ ]:
# check block_forward matches the phase 1 forward and the lm loss is sane
ids0 = torch.tensor([PREP[all_train[0][0]][0][:512]], device=dev)
with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
    T0 = ids0.size(1)
    _pm = torch.ones(1, T0, dtype=torch.bool, device=dev)
    _full, _win = build_masks(T0, _pm, False, ATTN_WINDOW)
    x_ = base.tok_emb(ids0)
    x_ref = x_
    for blk in base.layers:
        # our pass vs the original block
        x_ = block_forward(blk, x_, _win if blk.attn.window is not None else _full)
        x_ref = blk(x_ref)
    lg = base.lm_head(base.final_norm(x_)).float()
    lm_loss = F.cross_entropy(lg[0, :-1], ids0[0, 1:]).item()
    max_diff = (x_.float() - x_ref.float()).abs().max().item()
print(f"max diff vs phase 1 forward: {max_diff:.4f}")
print(f"lm loss on a train doc: {lm_loss:.3f}")
assert max_diff < 0.5
assert lm_loss < 6.0, "lm loss too high, weights and tokenizer probably do not match"
del _pm, _full, _win, x_, x_ref, lg

## Windows, dev split and optimizer

In [ ]:
names_index = {n: i for i, n in enumerate(TAG_NAMES)}
# dev docs come from the end of fold2
fold2_sorted = sorted(fold2, key=lambda r: r[0])
dev_rows = fold2_sorted[-DEV_COUNT:]  # last docs of fold2 are the dev set
dev_ids = {r[0] for r in dev_rows}
# train on fold1 and the rest of fold2
train_rows = list(fold1) + [r for r in fold2 if r[0] not in dev_ids]
if SMOKE:
    train_rows, dev_rows, EPOCHS = train_rows[:4], dev_rows[:2], 1
print("train docs", len(train_rows), "| dev docs", [r[0] for r in dev_rows])
prepare(train_rows + dev_rows + test_rows)

# (token ids, tag ids) for every window
train_windows = []
for doc_id, text, ents in train_rows:
    ids, offs = PREP[doc_id]
    tags = bio_tags(offs, ents)
    # cut every doc into overlapping windows
    for a, b in window_ranges(len(ids), WIDTH, STRIDE):
        train_windows.append((ids[a:b], tags[a:b]))
print(f"train windows: {len(train_windows)} (width {WIDTH}, stride {STRIDE})")

# class weights, only used when USE_CRF is False
tag_counts = np.ones(K)
for _ids, tgs in train_windows:
    for t in tgs:
        tag_counts[t] += 1
# rarer tags get more weight, capped by WEIGHT_CAP
w = 1.0 / np.sqrt(tag_counts)
w = w * (K / w.sum())
w = np.minimum(w, np.median(w) * WEIGHT_CAP)
cls_weights = torch.tensor(w, dtype=torch.float32, device=dev)


def collate(items):
    L = max(len(i) for i, _ in items)
    ids = torch.zeros(len(items), L, dtype=torch.long)
    # pad with id 0, target -100 (ignored) and mask False
    tgt = torch.full((len(items), L), -100, dtype=torch.long)
    pm = torch.zeros(len(items), L, dtype=torch.bool)
    for r, (i, t) in enumerate(items):
        ids[r, :len(i)] = torch.tensor(i)
        tgt[r, :len(t)] = torch.tensor(t)
        pm[r, :len(i)] = True
    return ids.to(dev), tgt.to(dev), pm.to(dev)


use_amp = dev.type == "cuda"


def make_run(exp):
    bb = fresh_base()
    tg = build_tagger(exp["method"], bb, exp["rank"], exp["alpha"], USE_GC or exp["method"] == "full")
    bb_params = [p for n, p in tg.named_parameters() if p.requires_grad and n.startswith("backbone.")]
    hd_params = [p for n, p in tg.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    groups = [{"params": hd_params, "lr": HEAD_LR}] + ([{"params": bb_params, "lr": exp["lr"]}] if bb_params else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    steps_per_epoch = math.ceil(len(train_windows) / BATCH)
    total_steps = steps_per_epoch * EPOCHS
    warm = max(1, int(WARMUP_FRAC * total_steps))
    # linear warmup, then linear decay down to 5%
    sch = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else max(0.05, (total_steps - s) / max(1, total_steps - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    print(f"steps/epoch {steps_per_epoch}, total {total_steps}, warmup {warm}")
    return tg, opt, sch, scaler


## Inference

In [ ]:
# logits for a whole doc, overlapping windows averaged with more weight in the middle
@torch.no_grad()
def doc_logits(tg, ids):
    tg.eval()
    n = len(ids)
    ids_t = torch.tensor(ids, dtype=torch.long)
    # acc = weighted sum of logits, wsum = sum of the weights
    acc = torch.zeros(n, K, device=dev)
    wsum = torch.zeros(n, 1, device=dev)
    ranges = window_ranges(n, WIDTH, STRIDE)
    for i in range(0, len(ranges), INFER_BATCH):
        chunk = ranges[i:i + INFER_BATCH]
        L = max(b - a for a, b in chunk)
        x = torch.zeros(len(chunk), L, dtype=torch.long)
        pm = torch.zeros(len(chunk), L, dtype=torch.bool)
        for r, (a, b) in enumerate(chunk):
            x[r, :b - a] = ids_t[a:b]
            pm[r, :b - a] = True
        x, pm = x.to(dev), pm.to(dev)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
            out = tg(x, pm)
        out = out.float()
        for r, (a, b) in enumerate(chunk):
            m = b - a
            # triangle weight, highest in the middle of the window
            wt = torch.minimum(torch.arange(1, m + 1, device=dev), torch.arange(m, 0, -1, device=dev)).float().unsqueeze(1)
            acc[a:b] += out[r, :m] * wt
            wsum[a:b] += wt
    # weighted average of the overlapping windows
    return (acc / wsum.clamp(min=1e-6)).cpu().numpy()


# transitions: learned ones from the crf, or hard rules without it
def get_trans(tg):
    if tg.crf is not None:
        return tg.crf.transition().detach().float().cpu().numpy().astype(np.float64)
    return HARD_TRANS


# (doc_id, text, offsets, logits) for every doc
def build_cache(tg, rows, desc="scoring"):
    return [(d, t, PREP[d][1], doc_logits(tg, PREP[d][0])) for d, t, _ in tqdm(rows, desc=desc, leave=False)]


# logits -> best path -> spans
def decode_cache(cache, trans, o_delta=0.0, trim_mode=0):
    return {d: decode_spans(t, offs, best_path(lg, trans, USE_CRF, o_delta), trim_mode) for d, t, offs, lg in cache}


def evaluate_dev(tg):
    cache = build_cache(tg, dev_rows)
    return strict_scores(dev_rows, decode_cache(cache, get_trans(tg), 0.0, TRIM_MODE))

## Training loop

In [ ]:
def distill_loss(student_em, teacher_em, pad_mask):
    s_logp = F.log_softmax(student_em / KD_TEMP, dim=-1)
    t_prob = F.softmax(teacher_em / KD_TEMP, dim=-1)
    kl = (t_prob * (torch.log(t_prob.clamp(min=1e-9)) - s_logp)).sum(-1)
    mf = pad_mask.float()
    return (kl * mf).sum() / mf.sum().clamp(min=1) * KD_TEMP ** 2


# keeps the checkpoint with the best dev f1, stops after PATIENCE epochs without improvement
def run_training(tg, opt, sched, scaler, ckpt_file, epochs, patience, batch, tag, teacher=None):
    trainable = [p for p in tg.parameters() if p.requires_grad]
    best, bad, t0 = -1.0, 0, time.time()
    for ep in range(1, epochs + 1):
        tg.train()
        # new random order every epoch
        order = np.random.permutation(len(train_windows)); tot, nb = 0.0, 0
        bar = tqdm(range(0, len(order), batch), desc=f"{tag} ep{ep}/{epochs}", leave=False)
        for bi in bar:
            ids, tgt, pm = collate([train_windows[i] for i in order[bi:bi + batch]])
            with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                em = tg(ids, pm)
            # loss is computed in fp32
            em = em.float()
            # crf loss, or weighted cross entropy when there is no crf
            if tg.crf is not None:
                loss = tg.crf.nll(em, tgt, pm)
            else:
                loss = F.cross_entropy(em.reshape(-1, K), tgt.reshape(-1), weight=cls_weights, ignore_index=-100)
            if teacher is not None:
                with torch.no_grad(), torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=use_amp):
                    t_em = teacher(ids, pm)
                loss = KD_WEIGHT * loss + (1 - KD_WEIGHT) * distill_loss(em, t_em.float(), pm)
            opt.zero_grad(set_to_none=True)
            # unscale first so the clipping sees the real gradients
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(trainable, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += float(loss.detach()); nb += 1
            bar.set_postfix(loss=f"{tot / nb:.4f}")
        # dev score after every epoch
        sc = evaluate_dev(tg)
        print(f"  [{tag}] ep {ep}: loss {tot / nb:.4f} | dev strict P {sc['p']:.4f} R {sc['r']:.4f} F1 {sc['f1']:.4f} | {(time.time() - t0) / 60:.1f} min")
        if sc["f1"] > best:
            best, bad = sc["f1"], 0
            torch.save({"model": tg.state_dict(), "dev_f1": best, "epoch": ep}, ckpt_file)
        else:
            bad += 1
            if bad >= patience:
                print("  early stop"); break
    # go back to the best epoch
    tg.load_state_dict(torch.load(ckpt_file, map_location=dev)["model"])
    return best


def get_outputs(tg):
    trans = get_trans(tg)
    dev_c = build_cache(tg, dev_rows, "dev logits")
    test_c = build_cache(tg, test_rows, "test logits")
    return dev_c, test_c, trans

## Decode and submission

In [ ]:
def write_submission(exp, dev_c, test_c, trans):
    modes = [TRIM_MODE] + [m for m in (0, 1, 2) if m != TRIM_MODE]
    best_sc, best_cfg = None, None
    for delta in DELTAS:
        for mode in modes:
            sc = strict_scores(dev_rows, decode_cache(dev_c, trans, delta, mode))
            if best_sc is None or sc["f1"] > best_sc["f1"] + 1e-9:
                best_sc, best_cfg = sc, (delta, mode)
    delta, mode = best_cfg
    print(f"[{exp['name']}] o_delta {delta} | trim {mode}")
    print(f"[{exp['name']}] dev strict macro-F1 {best_sc['f1']:.4f}")
    print_scores(best_sc)

    preds = decode_cache(test_c, trans, delta, mode)
    payload = {d: [[int(s), int(e), lab] for s, e, lab in preds[d]] for d, _, _ in test_rows}
    assert set(payload) == {d for d, _, _ in test_rows}
    pred_path = os.path.join(OUT_DIR, f"prediction_{exp['name']}.json")
    with open(pred_path, "w", encoding="utf-8") as fh:
        json.dump(payload, fh, ensure_ascii=False)
    zip_path = f"/kaggle/working/submission_exp{exp['num']}_{exp['name']}.zip"
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")
    print(f"wrote {zip_path} | {sum(len(v) for v in payload.values())} spans over {len(payload)} test docs")


## Train

In [ ]:
del base
gc.collect(); torch.cuda.empty_cache()

teacher = None
for exp in EXPERIMENTS:
    print(f"\n===== exp {exp['num']}: {exp['name']} =====")
    tagger, optimizer, sched, scaler = make_run(exp)
    ckpt_file = os.path.join(OUT_DIR, f"best_{exp['name']}.pt")
    run_training(tagger, optimizer, sched, scaler, ckpt_file, EPOCHS, PATIENCE, BATCH, exp["name"],
                 teacher=teacher if exp.get("kd") else None)
    dev_c, test_c, trans = get_outputs(tagger)
    write_submission(exp, dev_c, test_c, trans)
    if exp["method"] == "full":
        teacher = tagger
        teacher.eval()
        for p in teacher.parameters():
            p.requires_grad_(False)
    del tagger, optimizer, sched, scaler, dev_c, test_c
    gc.collect(); torch.cuda.empty_cache()

del teacher
gc.collect(); torch.cuda.empty_cache()
